In [1]:
from dotenv import load_dotenv
load_dotenv() # take environment variables from .env.

True

In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
    # api_key="...",  # if you prefer to pass api key in directly instaed of using env vars
    # base_url="...",
    # organization="...",
    # other params...
)
prompt = PromptTemplate(
        template="Explain about {topic} in 100 words.",
        input_variables=["topic"]
    )

chain_without_parser = prompt | model
result_without = chain_without_parser.invoke({"topic": "Large Language Models"})
print(result_without)

content='Large Language Models (LLMs) are advanced AI systems trained on massive datasets of text and code. Utilizing deep learning, particularly the transformer architecture, they learn patterns, grammar, and context to understand and generate human-like language. With billions of parameters, LLMs can perform diverse tasks like answering questions, writing articles, summarizing documents, translating languages, and even generating code. They operate by predicting the most probable next word in a sequence, making them incredibly versatile. Their ability to process and create coherent, contextually relevant text has revolutionized fields from customer service to content creation, fundamentally changing how we interact with information and technology.' additional_kwargs={} response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': []} id='run--05f1de28-068c-4661-9c7c-1e07202b2249-0' usage_m

In [3]:
parser = StrOutputParser()
chain_with_parser = prompt | model | parser
result_with = chain_with_parser.invoke({"topic": "Large Language Models"})
print(result_with)

Large Language Models (LLMs) are advanced AI systems trained on massive datasets of text and code. Utilizing deep learning, particularly the transformer architecture, they learn patterns, grammar, and context to understand and generate human-like language. With billions of parameters, LLMs can perform diverse tasks like answering questions, writing articles, summarizing documents, translating languages, and even generating code. They operate by predicting the most probable next word in a sequence, making them incredibly versatile. Their ability to process and create coherent, contextually relevant text has revolutionized fields from customer service to content creation, fundamentally changing how we interact with information and technology.


In [4]:
prompt_without_parser = PromptTemplate(
        template="Analyze the movie {movie_title} and provide details about genre, rating, and key themes.",
        input_variables=["movie_title"]
    )

In [5]:
chain_without_parser = prompt_without_parser | model
result_without = chain_without_parser.invoke({"movie_title": "The Matrix"})
print(result_without)

content='*The Matrix*, released in 1999, is a groundbreaking science fiction action film written and directed by the Wachowskis (Lana and Lilly Wachowski). It revolutionized filmmaking with its innovative visual effects and deeply philosophical narrative, leaving an indelible mark on popular culture.\n\n---\n\n### Genre\n\n*The Matrix* is primarily a **Science Fiction** film, but it masterfully blends elements from several other genres, creating a unique and influential cinematic experience:\n\n1.  **Science Fiction:** At its core, it explores futuristic concepts like artificial intelligence, virtual reality, dystopian futures, and the nature of consciousness.\n2.  **Action:** It\'s renowned for its highly stylized and innovative action sequences, heavily influenced by Hong Kong martial arts cinema (especially wire fu) and anime. The "bullet time" effect became iconic.\n3.  **Cyberpunk:** It features a bleak, high-tech, low-life future where powerful corporations/machines control socie

In [6]:
from langchain_core.output_parsers import JsonOutputParser

In [7]:
parser = JsonOutputParser()
format_instructions = parser.get_format_instructions()


prompt_with_parser = PromptTemplate(
    template="""Analyze the movie {movie_title} and provide details about genre, rating, and key themes.
    {format_instructions}""",
    input_variables=["movie_title"],
    partial_variables={"format_instructions": format_instructions}
    )
    
chain_with_parser = prompt_with_parser | model | parser
result_with = chain_with_parser.invoke({"movie_title": "The Matrix"})
print(result_with)

{'title': 'The Matrix', 'year': 1999, 'genre': ['Science Fiction', 'Action', 'Cyberpunk', 'Philosophical', 'Martial Arts'], 'rating': {'us': 'R', 'reason': 'For sci-fi violence and some language'}, 'key_themes': [{'name': 'Reality vs. Illusion', 'description': "The central premise revolves around the nature of reality, questioning what is real and what is a simulated construct. The 'red pill or blue pill' choice epitomizes this theme."}, {'name': 'Choice vs. Fate/Determinism', 'description': "Characters constantly grapple with the idea of free will versus a predetermined path. Neo's journey as 'The One' explores whether his destiny is fixed or if he actively chooses to become it."}, {'name': 'Rebellion and Freedom', 'description': "The film depicts humanity's struggle against an oppressive machine intelligence, fighting for liberation and the right to self-determination."}, {'name': 'Technology and Humanity', 'description': 'It explores the potential dangers of advanced artificial inte

In [8]:
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import List

In [9]:
# Define Pydantic model
class Person(BaseModel):
    name: str = Field(description="Person's full name")
    age: int = Field(description="Person's age in years", ge=0, le=150)
    occupation: str = Field(description="Person's job or profession")
    skills: List[str] = Field(description="List of person's key skills")

In [10]:
prompt_without_parser = PromptTemplate(
    template="Create a fictional person profile for a {profession}.",
    input_variables=["profession"]
)

chain_without_parser = prompt_without_parser | model
result_without = chain_without_parser.invoke({"profession": "data scientist"})

print("BEFORE PARSER:")
print("Prompt used:", prompt_without_parser.invoke({"profession": "data scientist"}))
print(f"Type: {type(result_without)}")
print(f"result_without: {result_without}")
print()

# WITH PARSER - Modified prompt with format instructions
parser = PydanticOutputParser(pydantic_object=Person)
format_instructions = parser.get_format_instructions()


prompt_with_parser = PromptTemplate(
    template="Create a fictional person profile for a {profession}.\n{format_instructions}",
    input_variables=["profession"],
    partial_variables={"format_instructions": format_instructions}
)

chain_with_parser = prompt_with_parser | model | parser
result_with = chain_with_parser.invoke({"profession": "data scientist"})

print("AFTER PARSER:")
print("Prompt used:", prompt_with_parser.invoke({"profession": "data scientist"}))
print(f"Type: {type(result_with)}")
print(f"result_with: {result_with}")

BEFORE PARSER:
Prompt used: text='Create a fictional person profile for a data scientist.'
Type: <class 'langchain_core.messages.ai.AIMessage'>
result_without: content='## Fictional Person Profile: Dr. Aris Thorne\n\n---\n\n**Name:** Dr. Aris Thorne\n**Title:** Lead Data Scientist, AI Ethics & Interpretability\n**Company:** Aethel Analytics (a leading firm specializing in ethical AI solutions for healthcare and finance)\n**Location:** Boston, MA\n**Age:** 36\n**Nationality:** American\n\n---\n\n### Professional Summary\n\nDr. Aris Thorne is a highly accomplished and ethically driven Lead Data Scientist with a unique interdisciplinary background spanning cognitive neuroscience, applied statistics, and computer science. At Aethel Analytics, he spearheads initiatives focused on developing transparent, fair, and robust AI models, particularly in high-stakes domains like medical diagnostics and financial risk assessment. Aris is passionate about bridging the gap between complex machine lear